<a href="https://colab.research.google.com/github/Nakib-Nasrullah/Dengue_Research/blob/main/Dengue_best_math_equa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Data handling and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Datasets/Dengue_final_dataset_updated.csv')
print("Dataset loaded successfully!!")
print(f"\n Dataset shape: {df.shape}")

Mounted at /content/drive
Dataset loaded successfully!!

 Dataset shape: (23360, 23)


In [4]:
# Convert date column
df['date'] = pd.to_datetime(df['Date'])

# Sort by date (CRITICAL)
df = df.sort_values('date').reset_index(drop=True)

# Basic check
print(df.head())

       Date    District  Max_Temp  Min_Temp  Avg_Temp  Humidity  Rainfall  \
0  1/1/2025       Dhaka     30.04     20.69     25.36     62.83      2.46   
1  1/1/2025   Netrokona     25.61     19.34     22.47     67.60      2.78   
2  1/1/2025   Sirajganj     25.02     17.17     21.09     54.53      0.54   
3  1/1/2025   Madaripur     29.36     20.40     24.88     50.32      1.64   
4  1/1/2025  Jhalokathi     35.23     29.54     32.38     55.68      1.29   

   Rainfall_Lag_7  Rainfall_Lag_14  Rainfall_Lag_21  ...  Temp_Lag_30  \
0             0.0              0.0              0.0  ...          0.0   
1             0.0              0.0              0.0  ...          0.0   
2             0.0              0.0              0.0  ...          0.0   
3             0.0              0.0              0.0  ...          0.0   
4             0.0              0.0              0.0  ...          0.0   

   Humidity_Lag_7  Humidity_Lag_14  Humidity_Lag_21  Humidity_Lag_30  \
0             0.0         

# **Create Gamma weights**

In [5]:
from scipy.stats import gamma

def create_weights(lag_min=7, lag_max=35, mean=18, sd=6):
    tau = np.arange(lag_min, lag_max + 1)

    # Convert mean & sd → gamma parameters
    shape = (mean / sd) ** 2
    scale = (sd ** 2) / mean

    weights = gamma.pdf(tau, a=shape, scale=scale)

    # Normalize
    weights = weights / weights.sum()

    return tau, weights


# **Compute infectious pressure**

In [6]:
def compute_infectious_pressure(cases, tau, weights):
    I = np.zeros(len(cases))

    for t in range(len(cases)):
        val = 0
        for i, lag in enumerate(tau):
            if t - lag >= 0:
                val += weights[i] * cases[t - lag]
        I[t] = val

    return I